### 나이브 베이즈 (Naive Bayes)
- 확률 기반 머신러닝 분류 알고리즘의 대표격
- 나이브 베이즈 분류 알고리즘은 데이터를 나이브(단순)하게 독립적인 사건으로 가정.
- 이 독립 사건들을 베이즈 이론에 대입시켜 가장 높은 확률 레이블로 분류를 실행하는 알고리즘

#### 가우시안 나이브 베이즈를 이용한 붓꽃 분류

In [58]:
import pandas as pd
df = pd.read_csv("../Data/iris.csv")
df.head()

,SepalLength,SepalWidth,PetalLength,PetalWidth,Name
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


In [59]:
from sklearn.model_selection import train_test_split

train_data, test_data, train_target, test_target = \
  train_test_split(
    df.iloc[:,:4],
    df.iloc[:,4],
    random_state=42,
    stratify=df.iloc[:,4],
    test_size=0.2
  )

In [60]:
from sklearn.naive_bayes import GaussianNB
clf = GaussianNB()
clf.fit(train_data, train_target)
print("Train:", clf.score(train_data,train_target))
print("Test :", clf.score(test_data,test_target))

Train: 0.9583333333333334
Test : 0.9666666666666667


---
### 베르누이 나이브 베이즈를 활용한 스팸 분류

In [14]:
df = pd.read_csv("../Data/email_train.csv")
df

,email title,spam
0,free game only today,True
1,cheapest flight deak,True
2,limited time offer only today only today,True
3,today meeting schedule,False
4,your flight schedule attached,False
5,your credit card statement,False


In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   email title  6 non-null      object
 1   spam         6 non-null      bool  
dtypes: bool(1), object(1)
memory usage: 186.0+ bytes


In [18]:
df.shape

(6, 2)

### 데이터 다듬기

In [19]:
df['label'] = df['spam'].map({True:1, False:0}) # map은 하나씩 읽어들여온다.

In [20]:
df

,email title,spam,label
0,free game only today,True,1
1,cheapest flight deak,True,1
2,limited time offer only today only today,True,1
3,today meeting schedule,False,0
4,your flight schedule attached,False,0
5,your credit card statement,False,0


In [21]:
df_x = df['email title']
df_y = df['label']

In [22]:
df_x

0                        free game only today
1                        cheapest flight deak
2    limited time offer only today only today
3                      today meeting schedule
4               your flight schedule attached
5                  your credit card statement
Name: email title, dtype: object

In [23]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import BernoulliNB
import numpy as np
np.random.seed(5)

In [24]:
# BernoulliNB 는 0하고 1밖에 모른다.
cv = CountVectorizer(binary=True) # 1하고 0밖에 없다.
x_traincv = cv.fit_transform(df_x)

In [30]:
encoded_input = x_traincv.toarray()
encoded_input

array([[0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0],
       [0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0],
       [1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1],
       [0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1]])

In [29]:
cv.get_feature_names_out()

array(['attached', 'card', 'cheapest', 'credit', 'deak', 'flight', 'free',
       'game', 'limited', 'meeting', 'offer', 'only', 'schedule',
       'statement', 'time', 'today', 'your'], dtype=object)

In [ ]:
cv.inverse_transform(encoded_input[0].reshape(1,-1)) # -1은 알아서 하라는 뜻

[array(['free', 'game', 'only', 'today'], dtype='<U9')]

In [34]:
bnb = BernoulliNB()
bnb.fit(x_traincv, df_y)

BernoulliNB()

In [35]:
### Test Data
test_df = pd.read_csv("../Data/email_test.csv")
test_df

,email title,spam
0,free flight offer,True
1,hey traveler free flight deal,True
2,limited free game iffer,True
3,today flight schedule,False
4,your credit card attached,False
5,free credit card offer only today,False


In [39]:
test_df['label'] = test_df['spam'].map({True:1, False:0})
test_x = test_df['email title']
test_y = test_df['label']
x_testcv = cv.transform(test_x)

In [ ]:
print(bnb.score(x_traincv, df_y))
print(bnb.score(x_testcv, test_y))


1.0
0.8333333333333334


---
### 다항분포 나이브베이즈 영화리뷰 감정 분류
: 영화 리뷰에 다항분포 나이브 베이즈 분류를 활용하여 영화리뷰가 긍정인지 부정인지 분류

In [42]:
df = pd.read_csv("../Data/naive_movie.csv")
df.head()

,movie_review,type
0,this is great great movie. I will watch again,positive
1,I like this movie,positive
2,amazing movie in this year,positive
3,cool my boyfriend also said the movie is cool,positive
4,awesome of the awesome movie ever,positive


In [43]:
df.shape

(10, 2)

In [44]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   movie_review  10 non-null     object
 1   type          10 non-null     object
dtypes: object(2)
memory usage: 292.0+ bytes


In [47]:
df['label'] = df['type'].map({"positive":1, "negative":0})
df

,movie_review,type,label
0,this is great great movie. I will watch again,positive,1
1,I like this movie,positive,1
2,amazing movie in this year,positive,1
3,cool my boyfriend also said the movie is cool,positive,1
4,awesome of the awesome movie ever,positive,1
5,shame I wasted money and time,negative,0
6,regret on this move. I will never never what m...,negative,0
7,I do not like this movie,negative,0
8,I do not like actors in this movie,negative,0
9,boring boring sleeping movie,negative,0


In [50]:
df_x = df['movie_review']
df_y = df['label']

In [51]:
cv = CountVectorizer() # binary true를 하지 않는다
x_traincv=cv.fit_transform(df_x)
encoded_input = x_traincv.toarray()
encoded_input

array([[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 1, 0, 0, 0, 1, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1],
       [0, 0, 1, 0, 0, 0, 0, 1, 2, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0,
        0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,
        0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 2,
        0, 0, 1, 1, 0, 0, 0, 0, 2, 0, 0, 0, 1, 1, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0,
        1, 0, 0, 0, 0, 0, 0, 0

In [52]:
cv.get_feature_names_out()

array(['actors', 'again', 'also', 'amazing', 'and', 'awesome', 'boring',
       'boyfriend', 'cool', 'director', 'do', 'ever', 'from', 'great',
       'in', 'is', 'like', 'money', 'move', 'movie', 'my', 'never', 'not',
       'of', 'on', 'regret', 'said', 'shame', 'sleeping', 'the', 'this',
       'time', 'wasted', 'watch', 'what', 'will', 'year'], dtype=object)

In [53]:
from sklearn.naive_bayes import MultinomialNB

In [54]:
mnb = MultinomialNB()
mnb.fit(x_traincv,df_y)

MultinomialNB()

In [55]:
test_df = pd.read_csv("../Data/naive_movie_test.csv")
test_df['label'] = test_df['type'].map({"positive":1, "negative":0})
test_x = test_df['movie_review']
test_y = test_df['label']

In [56]:
x_testcv = cv.transform(test_x)

In [57]:
print(mnb.score(x_traincv, df_y))
print(mnb.score(x_testcv, test_y))


0.9
1.0
